In [1]:
import os
import json
import pandas as pd
from typing import Dict

from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
from autogen.coding import LocalCommandLineCodeExecutor

In [2]:
OPENDS_NAME             = "CAN"
TASK                    = "join"

# the dataset (stored as an archive, it's large)
DATASET_PATH            = f"../data/datasets/datasets_{OPENDS_NAME}.zip"

# the Ground Truth file with table paris (from LakeBench)
GT_PATH                 = f"../data/{TASK}/{OPENDS_NAME}_ground_truth.csv"

# the clusters of tables obtained by tables_clustering.ipynb
CLUSTERS_PATH           = f'../data/{TASK}/clusters/clusters_{OPENDS_NAME}.json'

# the directory where are stored for each table a small snapshot (<5) of its rows
REPR_ROWS_DIR_PATH      = f'../data/representative_rows/{OPENDS_NAME}'

# the queries created, in a CSV file with fields
# NL-question, NL-human-like-question, SQL-query, table1-id, table2-id
RESULTS_PATH            = f'../data/queries/{OPENDS_NAME}.csv'

# Used to select the representative rows (by now, they are just randomly sampled)
MAX_UNIQUE_VALUES       = 10

# the number of rows to consider as an example snapshot 
# passed to the generator agent
N_REPRESENTATIVE_ROWS   = 3

# the number of queries that we want to generate for each pair of table
N_QUERIES_PER_PAIR      = 3

In [24]:
if not os.path.isdir(os.path.dirname(RESULTS_PATH)):
    os.makedirs(os.path.dirname(RESULTS_PATH))

In [3]:
import unicodedata

def sanitize_string(s):
    """
    Replaces problematic characters in column names with underscores,
    normalizes accents, and strips spaces.
    """
    if not isinstance(s, str):
        return s
    
    # Normalize accents (e.g., é -> e)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('utf-8')
    # Replace problematic characters with underscores
    return s.replace('\n', '_').replace(' ', '_').replace('.', '_').replace('"', '_').strip()
    
def sanitize_all_strings(strings):
    """
    Replaces problematic characters in column names with underscores,
    normalizes accents, and strips spaces.
    """
    return list(map(sanitize_string, strings))

### Setup the Agents

In [4]:
os.environ['OLLAMA_HOST']   = 'http://localhost:11434'
os.environ['no_proxy']      = 'localhost,127.0.0.1'

In [5]:
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
config_list = [
    {
        "model": "llama-3.3-70b-versatile",
        "api_type": "groq",
        "price": [0, 0],
        "timeout": 1_200,
        "temperature": 0,
        "tags": [
            "llama3.3"
        ]
    }
]

In [45]:
# set to true to disable agents conversation on stdout
silent = True

In [46]:
executor = LocalCommandLineCodeExecutor(
    timeout=10,  # Timeout for each code execution in seconds.
    work_dir='./coding',  # Use the temporary directory to store the code files.
)

In [47]:
nl_generator = AssistantAgent(
    name="NLQuestionGenerator",
    system_message="""
        You are an SQL expert of joinable tables.
        You will take two joinable tables from a datalake, each containing five example entries for each column. 
        You are required to GENERATE a natural language question that necessitate joining these two tables. 
        DO NOT generate the answer.
        Report also the corresponding SQL code block.
        Be sure to use the correct column names in the SQL block, and put column names inside ``.
        Generate the natural language question. 
        DO NOT add any other info.
        Make sure that SQL is written in a way that always returns at least one result.
        Make sure that table names are inside ''.
    """,
    llm_config={
        'config_list': config_list, # filter_config(config_list, {'tag': ['llama3.3']}), 
        'cache_seed': None
    },
    is_termination_msg=lambda x: x.get("content", "").rstrip().endswith("TERMINATE"),
    silent=silent
)


In [48]:
user_proxy = UserProxyAgent(
    name="User_proxy",
    system_message="A human admin. Execute code when needed",
    code_execution_config={
        "last_n_messages": 3,
        "executor": LocalCommandLineCodeExecutor(work_dir="groupchat"),
    },
    human_input_mode="NEVER",
    is_termination_msg=lambda x: x.get("content", "").rstrip().endswith("TERMINATE"),
    max_consecutive_auto_reply=5,
    silent=silent
)

In [49]:
def check_termination(msg: Dict):
    if "tool_responses" not in msg:
        return False
    # json_str = msg["tool_responses"][0]["content"]
    content = msg["tool_responses"][0]["content"]

    # print(f'>>>>>>>>>>>> OCCHIO {json_str}')
    # obj = json.loads(json_str.replace('\'', '"'))
    # print(f'>>>>>>>>>>>> TUTTO BENE {obj}')
    # return "error" not in json_str or obj["error"] is None and obj["reward"] == 1
    return "error" not in content


sql_writer = AssistantAgent(
    "sql_writer",
    system_message="You are good at writing SQL queries. Always respond with a function call to execute_sql(). DO NOT write python code.",
    is_termination_msg=check_termination,
    llm_config={
        'config_list': config_list, 
        'cache_seed': None
    },
    silent=silent
)

In [50]:
import csv
import sqlite3
from typing import Annotated, Dict


@sql_writer.register_for_llm(description="Function for executing SQL query and returning a response")
@user_proxy.register_for_execution()
def execute_sql(
    llm_question        : Annotated[str, "Think about the corresponding natural language question to SQL."],
    human_like_question : Annotated[str, "Convert question without using technical or specific column names. \
                                          Instead, replace them with more relatable and descriptive terms to make the question feel intuitive and conversational."],
    reflection          : Annotated[str, "Think about what to do"],
    sql                 : Annotated[str, "SQL query"], 
    table1              : Annotated[str, "Name of the first table"], 
    table2              : Annotated[str, "Name of the second table"]
) -> Annotated[Dict[str, str], "Dictionary with keys 'result' and 'error'"]:
        connection = None
        error = None
        result = None
        
        try:
            # Connect to the SQLite database
            connection = sqlite3.connect('tmp.db')
            cursor = connection.cursor()
            
            # Execute the SQL query
            cursor.execute(sql)
            # the fetchall could give troubles with large results set
            # result = cursor.fetchall()
            
            result = cursor.fetchmany(10)
            # Commit changes if the query modifies the database
            # (This should not happens, the queries are just SELECT)
            # connection.commit()
            
        except sqlite3.Error as e:
            error = str(e)
        
        finally:
            if connection:
                connection.close()
        
        if error:
            return {
                "error": error,
                "wrong_result": result if result else "No result",
            }
        
        
        elif len(result) == 0:
            return {
                "error":  'Empty result',
                "result": "No result",
            }
            
        else:
            # save results to csv
            file_exists = os.path.isfile(RESULTS_PATH)

            with open(RESULTS_PATH, 'a', newline='') as csvfile:
                csv_writer = csv.writer(csvfile)
                if not file_exists:
                    csv_writer.writerow(['llm_question', 'human_like_question', 'sql', 'table1', 'table2'])  # Write header
            
                csv_writer.writerow([llm_question.strip('"'), human_like_question.strip('"'), sql, table1, table2])

            return {"result": result}

In [51]:
sql_rewriter = AssistantAgent(
    "rewriter",
    system_message="Given an SQL query and an error try to modify the sql to avoid the error. Write only the new sql. DO NOT write python code.",
    is_termination_msg=check_termination,
    llm_config={
        'config_list': config_list,
        'cache_seed': None
    },
    silent=silent
)

In [52]:
groupchat = GroupChat(
    agents=[sql_writer, user_proxy, sql_rewriter], 
    messages=[], 
    max_round=30, 
    speaker_selection_method="round_robin"
)

manager = GroupChatManager(
    groupchat=groupchat, 
    llm_config={
        'config_list': config_list,
        'cache_seed': None
    }, 
    system_message="When the task is completed say TERMINATE",
    silent=silent
)

### Generate the Questions

In [53]:
with open(CLUSTERS_PATH, 'r') as fr:
    clusters = json.load(fr)

clusters = clusters['clusters']

In [54]:
# take only small clusters, for example those with <=N tables inside
max_cluster_size = 4
small_clusters = list(filter(lambda d: len(d['cluster']) <= max_cluster_size, clusters))
print(f'Number of clusters with at most {max_cluster_size}: {len(small_clusters)}')

Number of clusters with at most 4: 16


In [55]:
# define a function to read the tables directly from the .zip file
import zipfile

def read_table(table_id):
    with zipfile.ZipFile(DATASET_PATH, 'r') as zip_tables:
        with zip_tables.open(f'datasets_{OPENDS_NAME}/{table_id}', 'r') as table_file:
            try:
                df = pd.read_csv(table_file, on_bad_lines='skip')
                df.columns = sanitize_all_strings(df.columns)
                for attribute in df.columns:
                    df[attribute] = df[attribute].apply(sanitize_string)
                return df
            except Exception as e:
                return f'Error on reading table {table_id}: {e}'

In [56]:
# function calling example
table_id = small_clusters[0]['cluster'][0]
read_table(table_id)

,Ref_Date,GEOGRAPHY,CHARACTERISTICS,EDUCATIONLEVEL,SEX,AGEGROUP,Value
0,1990,Canada,"Population_(x_1,000)","Total,_all_education_levels",Both_sexes,15_years_and_over,21214_7
1,1991,Canada,"Population_(x_1,000)","Total,_all_education_levels",Both_sexes,15_years_and_over,21533_3
2,1992,Canada,"Population_(x_1,000)","Total,_all_education_levels",Both_sexes,15_years_and_over,21820_2
3,1993,Canada,"Population_(x_1,000)","Total,_all_education_levels",Both_sexes,15_years_and_over,22092_9
4,1994,Canada,"Population_(x_1,000)","Total,_all_education_levels",Both_sexes,15_years_and_over,22367_7
...,...,...,...,...,...,...,...
36445,2010,Alberta,Employment_rate_(rate),Above_bachelor's_degree,Females,65_years_and_over,x
36446,2011,Alberta,Employment_rate_(rate),Above_bachelor's_degree,Females,65_years_and_over,30_3
36447,2012,Alberta,Employment_rate_(rate),Above_bachelor's_degree,Females,65_years_and_over,27_4
36448,2013,Alberta,Employment_rate_(rate),Above_bachelor's_degree,Females,65_years_and_over,25_4


In [57]:
gt = pd.read_csv(GT_PATH)
gt

,query_table,candidate_table
0,CAN_CSV0000000000024893.csv,CAN_CSV0000000000020982.csv
1,CAN_CSV0000000000019383.csv,CAN_CSV0000000000002092.csv
2,CAN_CSV0000000000016797.csv,CAN_CSV0000000000011318.csv
3,CAN_CSV0000000000005930.csv,CAN_CSV0000000000001161.csv
4,CAN_CSV0000000000004711.csv,CAN_CSV0000000000004697.csv
...,...,...
3315,CAN_CSV0000000000013788.csv,CAN_CSV0000000000001071.csv
3316,CAN_CSV0000000000013788.csv,CAN_CSV0000000000000660.csv
3317,CAN_CSV0000000000013788.csv,CAN_CSV0000000000013712.csv
3318,CAN_CSV0000000000013788.csv,CAN_CSV0000000000000857.csv


In [58]:
from sqlalchemy import Integer, String, Float, Boolean, DateTime

def map_dtype(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return Integer
    elif pd.api.types.is_float_dtype(dtype):
        return Float
    elif pd.api.types.is_bool_dtype(dtype):
        return Boolean
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return DateTime
    elif pd.api.types.is_string_dtype(dtype):
        return String
    else:
        raise ValueError(f"Unsupported dtype: {dtype}")

In [59]:
def import_csv_to_sqlite(df, table_name, db_connection):
    df.to_sql(table_name, db_connection, if_exists='replace', index=False)

In [60]:
from groq import BadRequestError
from sqlalchemy import MetaData, Table, Column, create_engine
from sqlalchemy.schema import CreateTable

import autogen
from tqdm import tqdm

if not os.path.isdir(REPR_ROWS_DIR_PATH):
    os.makedirs(REPR_ROWS_DIR_PATH)

# setup the SQLAlchemy engine
engine = create_engine('sqlite:///tmp.db')
metadata = MetaData()
conn = engine.connect()

old = []
tab = gt.drop_duplicates(subset=['query_table', 'candidate_table']).reset_index()

# Start logging with logger_type and the filename to log to
# logging_session_id = autogen.runtime_logging.start(logger_type="file", config={"filename": "runtime.log"})
# print("Logging session ID: " + str(logging_session_id))

for idx, row in tqdm(tab.iterrows(), total=tab.shape[0]):
    print(f'############ {idx} / {tab.shape[0]} ############')
    table1_id = tab[['query_table']].iloc[idx].values[0]
    table2_id = tab[["candidate_table"]].iloc[idx].values[0]

    # Load the representation of both tables 1 and 2 if the 
    # relative CSV files already exist, otherwise read the original
    # tables and sample few rows from them. Then, store the 
    # created snapshot
    if not os.path.exists(f'{REPR_ROWS_DIR_PATH}/{table1_id}'):
        table1 = read_table(table1_id)
        sample1 = table1.sample(N_REPRESENTATIVE_ROWS)
        sample1.to_csv(f'{REPR_ROWS_DIR_PATH}/{table1_id}', index=False)
        import_csv_to_sqlite(table1, table1_id.removesuffix('.csv'), conn)
    else:
        sample1 = pd.read_csv(f'{REPR_ROWS_DIR_PATH}/{table1_id}')
    
    if not os.path.exists(f'{REPR_ROWS_DIR_PATH}/{table2_id}'):
        table2 = read_table(table1_id)
        sample2 = table2.sample(N_REPRESENTATIVE_ROWS)
        sample2.to_csv(f'{REPR_ROWS_DIR_PATH}/{table2_id}', index=False)
        import_csv_to_sqlite(table2, table2_id.removesuffix('.csv'), conn)
    else:
        sample2 = pd.read_csv(f'{REPR_ROWS_DIR_PATH}/{table2_id}')

    # Now the '.csv' suffix is no longer needed
    table1_id = table1_id.removesuffix('.csv')
    table2_id = table2_id.removesuffix('.csv')

    # Create a table declaration for the two tables
    t1_sql = Table(table1_id, metadata, *[Column(col_name, map_dtype(dtype)) for col_name, dtype in sample1.dtypes.items()])
    t2_sql = Table(table2_id, metadata, *[Column(col_name, map_dtype(dtype)) for col_name, dtype in sample2.dtypes.items()])
    
    # Create the statements for CREATE TABLE
    s1 = str(CreateTable(t1_sql).compile(engine))
    s2 = str(CreateTable(t2_sql).compile(engine))
    
    # Try to generate the questions
    for i in range(N_QUERIES_PER_PAIR):
        try:
            resuts=manager.initiate_chat(
            nl_generator,
            message=f"""
                These are two joinable tables:
                ##########################
                Table '{table1_id}': {s1} 

                Example:
                {sample1}
                ##########################
                Table '{table2_id}': {s2} 

                Example:
                {sample2}
                ##########################
                
                make sure to be a lot different from old question:
                {old}
                You are required to GENERATE a natural language question that necessitate joining these two tables. 
                Do not generate the answer.
                Report also the corresponding SQL code block.
                Rembember that table 1 is {table1_id} and table 2 name is {table2_id}.
                Be sure to use the correct column and table names in the SQL block, and put column names inside ''.
                Generate the natural language question and do not add any other info.
                Do not translate columns to english if they are in french, use names as it is.
                Make sure that table names are inside ''.
                If the tables in your opinion do not do join say TERMINATE.
                WRITE ONLY NATURAL QUESTION:
            """,
            silent=silent
            )
        except BadRequestError:
            print('Fake bad tool call')
        
        gt = pd.read_csv(RESULTS_PATH)
        old = list(gt[(gt['table1'] == table1_id) & (gt['table2'] == table2_id)]['sql'].values)

# autogen.runtime_logging.stop()
conn.close()

  0%|          | 0/3320 [00:00<?, ?it/s]

############ 0 / 3320 ############


  0%|          | 0/3320 [00:00<?, ?it/s]


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01jf797321fmvbmd3xq4s9e3yq` service tier `on_demand` on : Limit 100000, Used 99925, Requested 1987. Please try again in 27m31.715s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': '', 'code': 'rate_limit_exceeded'}}

# ISSUE: How we can actually generate a benchmark? The mapping LakeBench-GroundTruth - OpenDataPortalsIDs is missing

What if we generate for each query some random word, query the API, then take pairs of tables and if these are joinable use them as potential ground truth to define new queries?

In [98]:
import requests

In [101]:
api_base_url = 'https://open.canada.ca/data/api/action'
# api_base_url = 'https://catalog.data.gov/api/3/action'

q = f'{api_base_url}/package_search?q=environment&rows=100'
# q = f'{api_base_url}/package_list'
# q = 'https://api.us.socrata.com/api/catalog/v1'
response = requests.get(q)

response.status_code

200

In [106]:
response.json()['result']['results'][0]['keywords']

{'fr': ['carte', 'carte demographique', 'qualité de vie'],
 'en': ['demographic maps', 'map', 'quality of life']}

In [80]:
for result in response.json()['result']['results']:
    # print('P >', result['title'])
    for resource in result['resources']:
        if resource['format'].upper() in {'CSV', 'ZIP'}:
            print('\tR >', result['title'], resource['id'], resource['name'], resource['description'])

	R > Electric Vehicle Population Data fa51be35-691f-45d2-9f3e-535877965e69 Comma Separated Values File 
	R > Air Quality f3ed1638-92da-4f88-bb6b-7d3940514574 Comma Separated Values File 
	R > Sugar-Sweetened Beverage Consumption in California Residents 327fe3fc-8ae2-471e-b059-0e036dc87df9 Sugar-Sweetened Beverage Consumption in California Residents, 2012/2013 (CSV) The mean servings/times sugar-sweetened beverages consumed daily by California residents. These data are from the 2013 California Dietary Practices Surveys (CDPS), 2012 California Teen Eating, Exercise and Nutrition Survey (CalTEENS), and 2013 California Children’s Healthy Eating and Exercise Practices Survey (CalCHEEPS). These surveys are now discontinued. Adults, adolescents, and children (with parental assistance) were asked about the sugar-sweetened beverages they drank over the previous 24 hour period. Child/Adolescent: Fruit and vegetable, beverage, and junk food consumption, along with physical activity, sedentary tim

In [29]:
response.json()['result']['results'][0]['resources']

[{'cache_last_updated': None,
  'cache_url': None,
  'created': '2017-01-26T12:46:50.568141',
  'data_quality': [],
  'datastore_active': False,
  'description': '',
  'format': 'JP2',
  'hash': '',
  'id': '8d185c97-c9b0-4ff4-ae89-9597edbce52e',
  'language': ['en', 'fr'],
  'last_modified': None,
  'metadata_modified': '2017-01-26T12:46:50.568141',
  'mimetype': None,
  'mimetype_inner': None,
  'name': 'Download the English JP2 File through HTTP',
  'name_translated': {'fr': 'Télécharger le fichier en format JPEG2000 Anglais via HTTP',
   'en': 'Download the English JP2 File through HTTP'},
  'package_id': 'dea5e91e-8893-11e0-8a89-6cf049291510',
  'position': 0,
  'resource_type': 'dataset',
  'state': 'active',
  'url': 'https://ftp.geogratis.gc.ca/pub/nrcan_rncan/raster/atlas_6_ed/eng/6462_the_37th_federal_election_2000.jp2',
  'url_type': None},
 {'cache_last_updated': None,
  'cache_url': None,
  'created': '2017-01-26T12:46:50.568178',
  'data_quality': [],
  'datastore_active'